<a href="https://colab.research.google.com/github/OmRohamare/stats_for_data/blob/main/outlier_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [2]:
import pandas as pd                    # load data, mean(), median(), mode()
import numpy as np                     # mean, median, weighted average
from scipy import stats                # trim_mean, gmean, hmean, mode
import matplotlib.pyplot as plt        # visualization
import seaborn as sns


In [3]:
cleaned = pd.read_csv('/content/drive/MyDrive/statsdata/house_sales_cleaned.csv', sep='\t')
cleaned

,DocumentDate,SalePrice,PropertyID,PropertyType,ym,zhvi_px,zhvi_idx,AdjSalePrice,NbrLivingUnits,SqFtLot,...,Bathrooms,Bedrooms,BldgGrade,YrBuilt,YrRenovated,TrafficNoise,LandVal,ImpsVal,ZipCode,NewConstruction
0,2014-09-16,280000,1000102,Multiplex,2014-09-01,405100,0.930836,300805.0,2,9373,...,3.00,6,7,1991,0,0,70000,229000,98002,False
1,2006-06-16,1000000,1200013,Single Family,2006-06-01,404400,0.929228,1076162.0,1,20156,...,3.75,4,10,2005,0,0,203000,590000,98166,True
2,2007-01-29,745000,1200019,Single Family,2007-01-01,425600,0.977941,761805.0,1,26036,...,1.75,4,8,1947,0,0,183000,275000,98166,False
3,2008-02-25,425000,2800016,Single Family,2008-02-01,418400,0.961397,442065.0,1,8618,...,3.75,5,7,1966,0,0,104000,229000,98168,False
4,2013-03-29,240000,2800024,Single Family,2013-03-01,351600,0.807904,297065.0,1,8620,...,1.75,4,7,1948,0,0,104000,205000,98168,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22662,2011-04-08,325000,9842300710,Single Family,2011-04-01,318700,0.732307,443803.0,1,5468,...,1.75,3,7,1951,0,0,201000,172000,98126,False
22663,2007-09-28,1580000,9845500010,Single Family,2007-09-01,433500,0.996094,1586196.0,1,23914,...,4.50,4,11,2000,0,1,703000,951000,98040,False
22664,2012-07-09,165000,9899200010,Single Family,2012-07-01,325300,0.747472,220744.0,1,11170,...,1.00,4,6,1971,0,0,92000,130000,98055,False
22665,2006-05-26,315000,9900000355,Single Family,2006-05-01,400600,0.920496,342207.0,1,6223,...,2.00,3,7,1939,0,0,103000,212000,98166,False


**Z Score**: Of given columns

In [13]:
cols_to_check = ['SalePrice','SqFtLot',"SqFtTotLiving"]



# Calculate Z-scores
z_df = (cleaned[cols_to_check] - cleaned[cols_to_check].mean()) / cleaned[cols_to_check].std()


# Filter: keep rows where all selected columns have |Z| <= 3
df_clean = cleaned[(z_df.abs() <= 3).all(axis=1)]

print(df_clean)

      DocumentDate  SalePrice  PropertyID   PropertyType          ym  zhvi_px  \
0       2014-09-16  -0.657715     1000102      Multiplex  2014-09-01   405100   
1       2006-06-16   1.419039     1200013  Single Family  2006-06-01   404400   
2       2007-01-29   0.683522     1200019  Single Family  2007-01-01   425600   
3       2008-02-25  -0.239480     2800016  Single Family  2008-02-01   418400   
4       2013-03-29  -0.773090     2800024  Single Family  2013-03-01   351600   
...            ...        ...         ...            ...         ...      ...   
22661   2012-04-02  -0.982208  9842300695  Single Family  2012-04-01   315200   
22662   2011-04-08  -0.527918  9842300710  Single Family  2011-04-01   318700   
22664   2012-07-09  -0.989419  9899200010  Single Family  2012-07-01   325300   
22665   2006-05-26  -0.556762  9900000355  Single Family  2006-05-01   400600   
22666   2007-01-09  -0.124105  9906000035  Single Family  2007-01-01   425600   

       zhvi_idx  AdjSalePri

**Modified Z Score**

 Uses median and MAD, which are not dragged around by extreme values.
 The 0.6745 constant makes the scale comparable to a normal z-score.
### Modified Z-Score Formula

$$M_i = \frac{0.6745 \cdot (x_i - \tilde{x})}{\text{MAD}}$$

**Where:**
- $x_i$: The individual observation
- $\tilde{x}$: The median of the dataset ($\text{median}(X)$)
- $\text{MAD}$: Median Absolute Deviation, defined as $\text{median}(|x_i - \tilde{x}|)$
- $0.6745$: Scaling factor approximating the standard deviation under a normal distribution

> **Outlier Threshold:** Points with $|M_i| > 3.5$ are commonly treated as potential outliers.




In [30]:
# 1. Calculate the Median Absolute Deviation (MAD)
# scale='normal' automatically applies the 0.6745 factor
mad = stats.median_abs_deviation(cleaned[cols_to_check], scale='normal')

# 2. Calculate the Median
median = cleaned[cols_to_check].median()

# 3. Calculate the Modified Z-Score
# (Using 0.6745 * (X - Median) / MAD)
modified_z_df = (cleaned[cols_to_check] - median) / mad
print("Modified Z-Scores:\n", modified_z_df)

print(modified_z_df)


Modified Z-Scores:
        SalePrice   SqFtLot  SqFtTotLiving
0      -0.815243  0.592428       0.612037
1       3.238452  3.532211       2.315748
2       1.802768  5.135283       0.187358
3       0.001126  0.386591       1.611281
4      -1.040448  0.387136      -0.237320
...          ...       ...            ...
22662  -0.561887 -0.472197      -0.537094
22663   6.503928  4.556759       3.509845
22664  -1.462708  1.082346      -1.049206
22665  -0.618188 -0.266361      -0.705716
22666   0.226331 -0.763368      -1.398942

[22667 rows x 3 columns]
       SalePrice   SqFtLot  SqFtTotLiving
0      -0.815243  0.592428       0.612037
1       3.238452  3.532211       2.315748
2       1.802768  5.135283       0.187358
3       0.001126  0.386591       1.611281
4      -1.040448  0.387136      -0.237320
...          ...       ...            ...
22662  -0.561887 -0.472197      -0.537094
22663   6.503928  4.556759       3.509845
22664  -1.462708  1.082346      -1.049206
22665  -0.618188 -0.266361    

In [31]:
outliers_mask = (modified_z_df.abs() > 3.5).any(axis=1)

# 5. Filter and print the original rows containing the outliers
outliers_df = cleaned[outliers_mask]

print(f"Total outliers found: {len(outliers_df)}")
print(outliers_df)

Total outliers found: 2674
      DocumentDate  SalePrice  PropertyID   PropertyType          ym  zhvi_px  \
1       2006-06-16   1.419039     1200013  Single Family  2006-06-01   404400   
2       2007-01-29   0.683522     1200019  Single Family  2007-01-01   425600   
6       2013-08-28  -0.520707     3800004  Single Family  2013-08-01   374300   
18      2013-06-05  -0.659157    11000040  Single Family  2013-06-01   366100   
19      2008-03-20  -0.311589    11000041  Single Family  2008-03-01   415100   
...            ...        ...         ...            ...         ...      ...   
22593   2007-11-27   1.995916  9829200560  Single Family  2007-11-01   428400   
22595   2014-11-18   2.538179  9829201020  Single Family  2014-11-01   410700   
22599   2007-03-06   3.928451  9831200075  Single Family  2007-03-01   429600   
22603   2010-05-10   5.673502  9831200500  Single Family  2010-05-01   347800   
22663   2007-09-28   3.091980  9845500010  Single Family  2007-09-01   433500   



#Interquartile Range


In [27]:
q1,q3=cleaned[cols_to_check].quantile(0.25),cleaned[cols_to_check].quantile(0.75)
iqr=q3-q1
lower_bound=q1-1.5*iqr
upper_bound=q3+1.5*iqr


In [32]:
# Create a mask where rows are within bounds for ALL specified columns
is_inside_bounds = (cleaned[cols_to_check] >= lower_bound) & (cleaned[cols_to_check] <= upper_bound)
filtered_df = cleaned[is_inside_bounds.all(axis=1)]

"""It returns True only if every single checked column
in that row is within the lower and upper limits."""

print(filtered_df)


      DocumentDate  SalePrice  PropertyID   PropertyType          ym  zhvi_px  \
0       2014-09-16  -0.657715     1000102      Multiplex  2014-09-01   405100   
3       2008-02-25  -0.239480     2800016  Single Family  2008-02-01   418400   
4       2013-03-29  -0.773090     2800024  Single Family  2013-03-01   351600   
5       2009-03-30  -0.456097     3600090      Townhouse  2009-03-01   369800   
7       2007-05-24  -0.464462     3800009  Single Family  2007-05-01   432100   
...            ...        ...         ...            ...         ...      ...   
22661   2012-04-02  -0.982208  9842300695  Single Family  2012-04-01   315200   
22662   2011-04-08  -0.527918  9842300710  Single Family  2011-04-01   318700   
22664   2012-07-09  -0.989419  9899200010  Single Family  2012-07-01   325300   
22665   2006-05-26  -0.556762  9900000355  Single Family  2006-05-01   400600   
22666   2007-01-09  -0.124105  9906000035  Single Family  2007-01-01   425600   

       zhvi_idx  AdjSalePri

#OUTLIERs:


In [29]:
outside_bound = (cleaned[cols_to_check] < lower_bound) | (
    cleaned[cols_to_check] > upper_bound
)
outliersiqr_df = cleaned[outside_bound.all(axis=1)]
print(outliersiqr_df)

      DocumentDate  SalePrice  PropertyID   PropertyType          ym  zhvi_px  \
80      2010-10-25   3.475603    46100350  Single Family  2010-10-01   334700   
126     2006-11-27   1.822853    98000100  Single Family  2006-11-01   421200   
127     2010-08-17   2.210802    98000150  Single Family  2010-08-01   339900   
128     2006-06-28   2.816239    98000190  Single Family  2006-06-01   404400   
134     2006-08-11   1.693056    98000640  Single Family  2006-08-01   411100   
...            ...        ...         ...            ...         ...      ...   
21772   2011-10-28   1.419039  9412400165  Single Family  2011-10-01   312000   
21874   2007-05-08   5.024516  9471200425  Single Family  2007-05-01   432100   
22440   2007-01-13   3.149668  9578050170  Single Family  2007-01-01   425600   
22441   2008-03-05   3.870763  9578050170  Single Family  2008-03-01   415100   
22663   2007-09-28   3.091980  9845500010  Single Family  2007-09-01   433500   

       zhvi_idx  AdjSalePri

# 🎯 Consensus Outlier Detection: IQR + Modified Z-Score

Combining the **Interquartile Range (IQR)** method and the **Modified Z-Score** creates a strict, consensus-based outlier detection rule. A data point is flagged as an anomaly only if **both methods agree**.

---

### 💡 Why Combine Them?
* **Reduces False Positives:** Particularly effective in right-skewed distributions where natural long tails can trigger false alarms.
* **Dual Lenses:**
  * **IQR:** Non-parametric boundary based on data percentiles ($Q_1$ and $Q_3$).
  * **Modified Z-Score:** Measures distance from the median scaled by the Median Absolute Deviation ($\text{MAD}$).
* **Balances Stringency:** The standard $1.5 \times \text{IQR}$ threshold can be overly aggressive on heavy tails; requiring $|M_i| > 3.5$ acts as a verification check.

---

### 📐 The Mathematical Formulas

#### 1. Interquartile Range (IQR) Rule
$$\text{IQR} = Q_3 - Q_1$$

$$\text{Lower Limit} = Q_1 - 1.5 \cdot \text{IQR}$$

$$\text{Upper Limit} = Q_3 + 1.5 \cdot \text{IQR}$$

#### 2. Modified Z-Score ($M_i$)
$$M_i = \frac{0.6745 \cdot (x_i - \tilde{x})}{\text{MAD}}$$

$$\text{where } \text{MAD} = \text{median}(|x_i - \tilde{x}|)$$

---

### ⚙️ How the Consensus Approach Works
1. **Compute IQR Bounds:** Flag points where $x_i < \text{Lower Limit}$ or $x_i > \text{Upper Limit}$.
2. **Compute Modified Z-Scores:** Flag points where $|M_i| > 3.5$.
3. **Apply Intersection:** Combine flags using logical conditions:

| Strategy | Logic | When to Use |
| :--- | :--- | :--- |
| **Strict (Consensus)** | $\text{Is\_IQR} \land \text{Is\_ModZ}$ | Minimizes false positives; flags only extreme anomalies. |
| **Lenient (Union)** | $\text{Is\_IQR} \lor \text{Is\_ModZ}$ | Aggressive cleanup; ensures no edge cases escape. |

In [33]:
# ==========================================
# Consensus Outlier Detection (IQR + Mod-Z)
# ==========================================

# 1. IQR Outlier Mask (per-column)
q1 = cleaned[cols_to_check].quantile(0.25)
q3 = cleaned[cols_to_check].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

iqr_mask = (cleaned[cols_to_check] < lower_bound) | (cleaned[cols_to_check] > upper_bound)

# 2. Modified Z-Score Mask (per-column)
mad = stats.median_abs_deviation(cleaned[cols_to_check], scale="normal")
median = cleaned[cols_to_check].median()
modified_z = (cleaned[cols_to_check] - median) / mad

modz_mask = modified_z.abs() > 3.5

# 3. Combine IQR & Modified Z-Score per column (Consensus: AND)
# A value in a column is an outlier only if BOTH IQR and Mod-Z flag it
consensus_col_mask = iqr_mask & modz_mask

# 4. Row-level Flagging
# Flag any row where at least one of the analyzed columns is an agreed outlier
row_outlier_mask = consensus_col_mask.any(axis=1)

# 5. Extract Outliers and Clean Data
consensus_outliers_df = cleaned[row_outlier_mask]
df_cleaned_consensus = cleaned[~row_outlier_mask]

# Output diagnostics
print(f"Total rows in dataset:               {len(cleaned)}")
print(f"Outliers flagged by IQR (any col):   {iqr_mask.any(axis=1).sum()}")
print(f"Outliers flagged by Mod-Z (any col): {modz_mask.any(axis=1).sum()}")
print(f"Strict Consensus Outliers (AND):     {len(consensus_outliers_df)}")
print(f"Cleaned dataset rows remaining:      {len(df_cleaned_consensus)}")

# Display sample of consensus outliers
consensus_outliers_df.head()

Total rows in dataset:               22667
Outliers flagged by IQR (any col):   3151
Outliers flagged by Mod-Z (any col): 2674
Strict Consensus Outliers (AND):     2674
Cleaned dataset rows remaining:      19993


,DocumentDate,SalePrice,PropertyID,PropertyType,ym,zhvi_px,zhvi_idx,AdjSalePrice,NbrLivingUnits,SqFtLot,...,Bathrooms,Bedrooms,BldgGrade,YrBuilt,YrRenovated,TrafficNoise,LandVal,ImpsVal,ZipCode,NewConstruction
1,2006-06-16,1.419039,1200013,Single Family,2006-06-01,404400,0.929228,1076162.0,1,0.291888,...,3.75,4,10,2005,0,0,203000,590000,98166,True
2,2007-01-29,0.683522,1200019,Single Family,2007-01-01,425600,0.977941,761805.0,1,0.495447,...,1.75,4,8,1947,0,0,183000,275000,98166,False
6,2013-08-28,-0.520707,3800004,Single Family,2013-08-01,374300,0.860064,380785.0,1,0.787249,...,1.50,3,8,1961,0,0,165000,227000,98178,False
18,2013-06-05,-0.659157,11000040,Single Family,2013-06-01,366100,0.841222,332255.0,1,0.536747,...,3.00,3,6,2000,0,0,142000,189000,98001,False
19,2008-03-20,-0.311589,11000041,Single Family,2008-03-01,415100,0.953814,419369.0,1,0.426140,...,2.00,4,8,1960,0,0,139000,200000,98001,False
